# Solution: Training a Neural Forecasting Model

In [ ]:
import os
import warnings
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
US = {"Orange": "#EE7622"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel
from darts.utils.likelihood_models import QuantileRegression
from darts.models import ARIMA as DartsARIMA

In [ ]:
DATA_PATH = "../data/subscribers.csv"

Compare two N-BEATS configurations on subscriber growth to determine which window length produces better shape fit.

In [ ]:
# Load and split
df = pd.read_csv(DATA_PATH, parse_dates=["date"], index_col="date").asfreq("MS")
series = TimeSeries.from_series(df["subscribers"])
train, test = series.split_after(pd.Timestamp("2023-12-01"))

In [ ]:
# Short-window N-BEATS
short = NBEATSModel(
    input_chunk_length=12,
    output_chunk_length=6,
    n_epochs=50,
    likelihood=QuantileRegression([0.05, 0.25, 0.5, 0.75, 0.95]),
    random_state=42,
    pl_trainer_kwargs={"enable_progress_bar": True}
)
short.fit(train)
pred_short = short.predict(len(test), num_samples=100)

In [ ]:
# Long-window N-BEATS
long = NBEATSModel(
    input_chunk_length=36,
    output_chunk_length=12,
    n_epochs=50,
    likelihood=QuantileRegression([0.05, 0.25, 0.5, 0.75, 0.95]),
    random_state=42,
    pl_trainer_kwargs={"enable_progress_bar": True}
)
long.fit(train)
pred_long = long.predict(len(test), num_samples=100)

In [ ]:
# Convert to pandas for easier comparison
test_pd = test.pd_series()
short_pd = pred_short.quantile_timeseries(0.5).pd_series()
long_pd = pred_long.quantile_timeseries(0.5).pd_series()

rmse_short = np.sqrt(np.mean((test_pd - short_pd)**2))
rmse_long = np.sqrt(np.mean((test_pd - long_pd)**2))
print(f"Short-window RMSE: {rmse_short:.1f}")
print(f"Long-window RMSE: {rmse_long:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
train.plot(ax=ax, label="Train", color=UN["Black"])
test.plot(ax=ax, label="Test", color=UB["Brand Blue"])
short_fc = short.predict(len(test), num_samples=100)
long_fc = long.predict(len(test), num_samples=100)
short_fc.plot(ax=ax, label="Short Window", color=US["Orange"], low_quantile=0.05, high_quantile=0.95)
long_fc.plot(ax=ax, label="Long Window", color=UB["Medium Blue"], low_quantile=0.05, high_quantile=0.95)
ax.set_title("N-BEATS Window Comparison", fontsize=18, fontweight="bold")
ax.set_ylabel("Subscribers", fontsize=16)
ax.tick_params(axis="both", labelsize=14)
ax.legend()
plt.tight_layout()
plt.show()